# DeBCR API tutorial
## Deblur data using trained DeBCR model

This notebook shows how to restore low-quality microscopy data using DeBCR.

To achieve that you would need:
- **pre-processed input data** by normalization and patching;
- **trained DeBCR model** on the same-sized data as the input.

Please find on the DeBCR GitHub page links to:
- notebook tutorial on raw data pre-processing protocol; 
- samples, i.e. examples of pre-processed input data and trained DeBCR model weights.

In [ ]:
import debcr

### Load pre-processed input data

Set file path to your actual pre-processed input data (in NPZ or NPY format).

For sample data: ```/path/to/examples/DATASET_test.npz```

In [ ]:
data_filepath = '/path/to/load/data/input.npz'
data_filepath

Load input data

In [ ]:
data = debcr.data.load(data_filepath)

The example input data is provided as multi-array NPZ file, which contains two arrays: 
- "low" - input data (low-quality data to be improved)
- "gt" - ground-truth data for comparison

You can check the filenames as below

In [ ]:
data.files

and the respective array size:

In [ ]:
data["low"].shape, data["gt"].shape

### Visualize loaded input data

for sample data

In [ ]:
debcr.data.show(
    data = [data["low"], data["gt"]],
    slices = [250, 500, -1], # -1 is to pick a random slice
    titles = ['input', 'ground truth'],
    transpose=True
)

### Load trained DeBCR model

Set directory path to your actual trained DeBCR model weights.

For sample data: ```/path/to/examples/DATASET_DeBCR.zip```

In [ ]:
weights_dirpath = '/path/to/load/model/weights'
weights_dirpath

Load trained DeBCR model

In [ ]:
debcr_model = debcr.model.init(weights_dirpath, input_size=128)

The model you intend to use for prediction should had been trained on the same-sized along XY data, as the intended prediction input.

Show TensorFlow model info to see model structure details (e.g. to verify input size) 

In [ ]:
debcr_model.summary()

### Run DeBCR prediction

To run prediction simply pass the loaded trained DeBCR model and input data.

By changing the `batch_size` you can manage amount of data to be processed at once for GPU memory control purposes.

In [ ]:
data_pred = debcr.model.predict(eval_model=debcr_model, input_data=data['low'], batch_size=32)
data_pred.shape

### Visualize predictions

for sample data

In [ ]:
debcr.data.show(
    data = [data['low'], data_pred, data["gt"]],
    slices = [250, 500, -1],
    titles = ['input', 'prediction', 'ground truth']
)

or for custom data

In [ ]:
debcr.data.show(
    data = [data['low'], data_pred],
    slices = [-1, -1, -1],
    titles = ['input', 'prediction'],
    transpose=True
)

### Save patched predictions

Provide below the file path to save the obtained patched predictions in the NPZ format:

In [ ]:
data_pred_filepath = '/path/to/save/data/pred.npz'
debcr.data.write(data_pred_filepath, data=data_asmbl)

### (optional) Post-process to get the full-size predictions and save result

The stitching is only possible if the following parameters are known: patch overlap and patch count along XY.

Therefore,
- skip this step, if you used the provided test input data;
- do this step, if you started before with the raw data pre-processing and know the respective parameters.

The patch count along XY can be obtained by the dry-run of the API `debcr.data.crop` (see data pre-processing protocol).

In [ ]:
data_asmbl = debcr.data.stitch(data_pred, patch_num=(15,15), overlap=(0.5, 0.5))
data_asmbl.shape

Let's also visualize result of the stitching

In [ ]:
debcr.data.show([data_asmbl], slices=[50,70,90], cmap='gray', transpose=True)

Finally, provide the path to save the obtained full-size predictions in the TIF format:

In [ ]:
data_pred_filepath = '/path/to/save/data/pred.tif'
debcr.data.write(data_pred_filepath, data=data_asmbl)

Next step is applying the whole DeBCR pipeline to your own data.

Good luck with deblurring using DeBCR!